# Trabajo Integrador Final — Curso de Data Science
## UTN BA — Centro de E-Learning

**Alumno:** Franco Palladino  
**Dataset:** Default of Credit Card Clients (UCI Machine Learning Repository)  
**Problema:** Clasificación binaria — predecir si un cliente incumplirá el pago de su tarjeta de crédito el mes siguiente.

---

## 0. Instalación e importación de librerías

Se utilizan las librerías estándar del ecosistema de Data Science en Python: `pandas` y `numpy` para manipulación de datos, `matplotlib` y `seaborn` para visualización, y `scikit-learn` para preprocesamiento, modelado y evaluación. Adicionalmente se importa `scipy.stats` para los tests de hipótesis estadísticos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

# Configuración global de gráficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('muted')

print('Librerías cargadas correctamente ✓')

---
## 1. Selección del Dataset

### Dataset elegido
**Default of Credit Card Clients** — UCI Machine Learning Repository (Yeh & Lien, 2009).

### Descripción general
El dataset contiene información de **30.000 clientes** de una institución financiera de Taiwán, recopilada durante el año 2005. Incluye variables demográficas (sexo, nivel educativo, estado civil, edad), el límite de crédito otorgado, el historial de pagos de los últimos 6 meses (abril a septiembre), los montos facturados y los montos efectivamente pagados en ese período.

### Justificación de la elección
Se seleccionó este dataset por los siguientes motivos:

1. **Relevancia práctica:** la predicción del riesgo crediticio es uno de los problemas de clasificación más extendidos en la industria financiera. Bancos, fintechs y entidades reguladoras emplean modelos similares para la aprobación y monitoreo de líneas de crédito.
2. **Cumplimiento de requisitos:** el dataset supera ampliamente los mínimos exigidos por la consigna (más de 500 filas y más de 5 columnas), con 30.000 registros y 24 variables.
3. **Riqueza analítica:** combina variables categóricas y numéricas, presenta desbalance de clases y multicolinealidad, lo que permite aplicar un rango amplio de técnicas de limpieza, análisis y modelado.
4. **Disponibilidad pública:** se encuentra accesible tanto en el repositorio oficial de UCI como en Kaggle, lo cual garantiza la reproducibilidad del análisis.

In [ ]:
# Carga del dataset desde Kaggle (fuente más estable para Colab)
url = 'https://raw.githubusercontent.com/gastonstat/CreditCards/master/CreditCards.csv'

try:
    df = pd.read_csv(url)
    print(f'Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas')
except Exception as e:
    print(f'Error con la primera URL: {e}')
    # Fallback: cargar desde UCI directamente (formato .xls)
    url2 = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
    df = pd.read_excel(url2, header=1)
    print(f'Dataset cargado desde UCI: {df.shape[0]} filas, {df.shape[1]} columnas')

df.head()

---
## 2. Exploración de Datos y Diccionario de Variables

En esta sección se realiza una inspección inicial del dataset para comprender su estructura, tipos de datos y valores presentes. Esto permite diseñar las estrategias de limpieza y transformación adecuadas.

In [ ]:
# Estandarizar nombres de columnas a minúsculas y sin espacios
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('.', '_', regex=False))

# Renombrar la columna objetivo a un nombre corto y consistente
for posible in ['default_payment_next_month', 'default__payment__next__month',
                'default_payment_next_month', 'y']:
    if posible in df.columns:
        df.rename(columns={posible: 'default'}, inplace=True)
        break

# Si no se encontró, asumir que la última columna es el target
if 'default' not in df.columns:
    df.rename(columns={df.columns[-1]: 'default'}, inplace=True)

print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nColumnas del dataset:')
print(df.columns.tolist())

In [ ]:
print('=== INFORMACIÓN GENERAL DEL DATASET ===')
df.info()
print()
print('=== PRIMERAS 5 FILAS ===')
df.head()

In [ ]:
# Diccionario de variables
diccionario = pd.DataFrame({
    'Variable': [
        'id', 'limit_bal', 'sex', 'education', 'marriage', 'age',
        'pay_0 a pay_6', 'bill_amt1 a bill_amt6', 'pay_amt1 a pay_amt6', 'default'
    ],
    'Tipo': [
        'Identificador', 'Numérica continua', 'Categórica nominal',
        'Categórica ordinal', 'Categórica nominal', 'Numérica discreta',
        'Categórica ordinal', 'Numérica continua', 'Numérica continua',
        'Binaria (target)'
    ],
    'Descripción': [
        'Identificador único del cliente',
        'Límite de crédito otorgado (en dólares taiwaneses, NTD)',
        'Sexo (1 = Hombre, 2 = Mujer)',
        'Nivel educativo (1 = Posgrado, 2 = Universidad, 3 = Secundaria, 4 = Otros)',
        'Estado civil (1 = Casado/a, 2 = Soltero/a, 3 = Otros)',
        'Edad del cliente en años',
        'Estado de pago mensual de abril a septiembre 2005 (-1 = pago puntual, 1 a 9 = meses de retraso)',
        'Monto facturado en cada mes (abril a septiembre 2005, en NTD)',
        'Monto del pago realizado en cada mes (abril a septiembre 2005, en NTD)',
        'Variable objetivo: 1 = el cliente incumple el pago al mes siguiente, 0 = no incumple'
    ]
})

print('=== DICCIONARIO DE VARIABLES ===')
diccionario.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

---
## 3. Limpieza de Datos

La limpieza de datos comprende la detección y tratamiento de valores faltantes, duplicados, categorías no documentadas y la eliminación de variables irrelevantes para el modelado. Cada decisión se justifica en el contexto del problema.

In [ ]:
print('=== 3.1 VALORES FALTANTES ===')
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)

if nulos.sum() > 0:
    resumen_nulos = pd.DataFrame({'Faltantes': nulos[nulos > 0], '% del total': nulos_pct[nulos_pct > 0]})
    print(resumen_nulos)
else:
    print('No se detectaron valores faltantes en ninguna variable. ✓')

print()
print('=== 3.2 REGISTROS DUPLICADOS ===')
duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {duplicados}')
if duplicados > 0:
    df.drop_duplicates(inplace=True)
    print(f'Se eliminaron {duplicados} filas duplicadas. Nuevo tamaño: {df.shape[0]} filas.')
else:
    print('No se requiere eliminación de duplicados. ✓')

In [ ]:
# 3.3 Eliminación de la columna ID
# La columna ID es un identificador secuencial sin valor predictivo.
# Incluirla en el modelo introduciría ruido y podría generar sobreajuste.
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    print('Columna "id" eliminada: es un identificador sin valor predictivo. ✓')

print(f'Dimensiones actuales: {df.shape[0]} filas × {df.shape[1]} columnas')

In [ ]:
# 3.4 Verificación y limpieza de categorías no documentadas
print('=== VALORES ÚNICOS EN VARIABLES CATEGÓRICAS ===')
cat_cols = [c for c in ['sex', 'education', 'marriage'] if c in df.columns]

for col in cat_cols:
    vals = sorted(df[col].unique())
    print(f'  {col}: {vals}')

In [ ]:
# Tratamiento de categorías no documentadas

# EDUCATION: según la documentación oficial, las categorías válidas son:
#   1 = Posgrado, 2 = Universidad, 3 = Secundaria, 4 = Otros.
# Los valores 0, 5 y 6 no están documentados. Se agrupan en la categoría 4 (Otros)
# para no perder registros y mantener la coherencia del esquema.
if 'education' in df.columns:
    antes = df['education'].value_counts().sort_index()
    df['education'] = df['education'].replace({0: 4, 5: 4, 6: 4})
    despues = df['education'].value_counts().sort_index()
    print('EDUCATION — valores 0, 5 y 6 reasignados a categoría 4 (Otros):')
    print(f'  Antes:   {antes.to_dict()}')
    print(f'  Después: {despues.to_dict()}')
    print()

# MARRIAGE: la categoría 0 no está documentada.
# Se agrupa en la categoría 3 (Otros) por el mismo criterio.
if 'marriage' in df.columns:
    antes = df['marriage'].value_counts().sort_index()
    df['marriage'] = df['marriage'].replace({0: 3})
    despues = df['marriage'].value_counts().sort_index()
    print('MARRIAGE — valor 0 reasignado a categoría 3 (Otros):')
    print(f'  Antes:   {antes.to_dict()}')
    print(f'  Después: {despues.to_dict()}')

In [ ]:
# 3.5 Decisión sobre outliers numéricos
# En el contexto crediticio, los valores extremos de LIMIT_BAL, BILL_AMT y PAY_AMT
# son legítimos: existen clientes con límites muy altos o pagos excepcionales.
# Eliminar estos registros sesgaría el modelo hacia clientes promedio, perdiendo
# capacidad predictiva en los extremos de la distribución.

print('=== DECISIÓN SOBRE OUTLIERS ===')
if 'limit_bal' in df.columns:
    q1 = df['limit_bal'].quantile(0.25)
    q3 = df['limit_bal'].quantile(0.75)
    iqr = q3 - q1
    outliers_count = ((df['limit_bal'] < q1 - 1.5*iqr) | (df['limit_bal'] > q3 + 1.5*iqr)).sum()
    print(f'Outliers detectados en limit_bal (criterio IQR): {outliers_count} registros')

print('Decisión: NO se eliminan outliers numéricos.')
print('Justificación: los montos extremos son válidos en el contexto financiero y representan')
print('segmentos reales de la cartera de clientes (clientes premium vs. clientes de riesgo).')
print()
print(f'Dataset final para análisis: {df.shape[0]} filas × {df.shape[1]} columnas')

---
## 4. Análisis Estadístico con Visualizaciones

Se presenta un análisis estadístico descriptivo e inferencial de las principales variables del dataset, acompañado de visualizaciones que permiten identificar patrones y relaciones relevantes para el problema de clasificación.

In [ ]:
print('=== 4.1 ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS ===')
df.describe().round(2)

In [ ]:
# 4.2 Distribución de la variable objetivo
target_col = 'default'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico de torta
conteo = df[target_col].value_counts()
axes[0].pie(conteo.values,
            labels=['No incumple (0)', 'Incumple (1)'],
            autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'],
            startangle=90,
            explode=[0, 0.05],
            textprops={'fontsize': 11})
axes[0].set_title('Distribución de la variable objetivo', fontsize=13)

# Gráfico de barras
bars = axes[1].bar(['No incumple (0)', 'Incumple (1)'],
                   conteo.values,
                   color=['#4CAF50', '#F44336'],
                   edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, conteo.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', fontsize=11)
axes[1].set_title('Cantidad de clientes por clase', fontsize=13)
axes[1].set_ylabel('Cantidad de clientes')

plt.tight_layout()
plt.show()

pct_default = conteo.get(1, 0) / len(df) * 100
print(f'Proporción de incumplimiento: {pct_default:.1f}%')
print(f'Ratio de desbalance: {conteo.get(0, 0) / conteo.get(1, 1):.1f} : 1')

In [ ]:
# 4.3 Distribución de variables numéricas clave
num_vars = [c for c in ['limit_bal', 'age'] if c in df.columns]
titulos = {'limit_bal': 'Límite de crédito (NTD)', 'age': 'Edad (años)'}

if num_vars:
    fig, axes = plt.subplots(1, len(num_vars), figsize=(14, 5))
    if len(num_vars) == 1:
        axes = [axes]

    for i, col in enumerate(num_vars):
        sns.histplot(df[col], bins=40, kde=True, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Media: {df[col].mean():,.0f}')
        axes[i].axvline(df[col].median(), color='orange', linestyle='--', label=f'Mediana: {df[col].median():,.0f}')
        axes[i].set_title(f'Distribución de {titulos.get(col, col)}', fontsize=13)
        axes[i].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Asimetría
    for col in num_vars:
        skew = df[col].skew()
        tipo = 'simétrica' if abs(skew) < 0.5 else ('asimetría positiva (cola derecha)' if skew > 0 else 'asimetría negativa (cola izquierda)')
        print(f'{col}: skewness = {skew:.2f} → {tipo}')

In [ ]:
# 4.4 Tasa de incumplimiento por variables categóricas
cat_vars = [c for c in ['sex', 'education', 'marriage'] if c in df.columns]

labels_map = {
    'sex':       {1: 'Hombre', 2: 'Mujer'},
    'education': {1: 'Posgrado', 2: 'Universidad', 3: 'Secundaria', 4: 'Otros'},
    'marriage':  {1: 'Casado/a', 2: 'Soltero/a', 3: 'Otros'}
}

titulos_cat = {
    'sex': 'Sexo',
    'education': 'Nivel educativo',
    'marriage': 'Estado civil'
}

if cat_vars:
    fig, axes = plt.subplots(1, len(cat_vars), figsize=(15, 5))
    if len(cat_vars) == 1:
        axes = [axes]

    for i, col in enumerate(cat_vars):
        tasa = df.groupby(col)[target_col].mean() * 100
        etiquetas = [labels_map.get(col, {}).get(k, str(k)) for k in tasa.index]
        bars = axes[i].bar(etiquetas, tasa.values, color='coral', edgecolor='black', linewidth=0.5)
        for bar, val in zip(bars, tasa.values):
            axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f'{val:.1f}%', ha='center', fontsize=10)
        axes[i].set_title(f'Tasa de incumplimiento por {titulos_cat.get(col, col)}', fontsize=12)
        axes[i].set_ylabel('% de incumplimiento')
        axes[i].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()

In [ ]:
# 4.5 Límite de crédito según incumplimiento (boxplot)
if 'limit_bal' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Boxplot
    sns.boxplot(x=target_col, y='limit_bal', data=df,
                palette=['#4CAF50', '#F44336'], ax=axes[0])
    axes[0].set_xticklabels(['No incumple', 'Incumple'])
    axes[0].set_title('Límite de crédito según incumplimiento', fontsize=13)
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Límite de crédito (NTD)')

    # Violin plot
    sns.violinplot(x=target_col, y='limit_bal', data=df,
                   palette=['#4CAF50', '#F44336'], ax=axes[1], inner='quartile')
    axes[1].set_xticklabels(['No incumple', 'Incumple'])
    axes[1].set_title('Distribución del límite de crédito (violin plot)', fontsize=13)
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Límite de crédito (NTD)')

    plt.tight_layout()
    plt.show()

    # Test estadístico: diferencia de medias
    grupo_0 = df[df[target_col] == 0]['limit_bal']
    grupo_1 = df[df[target_col] == 1]['limit_bal']
    t_stat, p_value = stats.ttest_ind(grupo_0, grupo_1, equal_var=False)
    print(f'Test t de Welch (limit_bal entre grupos):')
    print(f'  Estadístico t = {t_stat:.2f}, p-valor = {p_value:.2e}')
    print(f'  → La diferencia es {"estadísticamente significativa" if p_value < 0.05 else "no significativa"} (α = 0.05)')

In [ ]:
# 4.6 Distribución de edad por grupo
if 'age' in df.columns:
    plt.figure(figsize=(10, 5))
    for grupo, color, label in [(0, '#4CAF50', 'No incumple'), (1, '#F44336', 'Incumple')]:
        sns.kdeplot(df[df[target_col] == grupo]['age'], color=color, label=label, fill=True, alpha=0.3)
    plt.title('Distribución de edad según incumplimiento', fontsize=13)
    plt.xlabel('Edad (años)')
    plt.ylabel('Densidad')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 4.7 Mapa de correlación
plt.figure(figsize=(14, 10))
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8},
            vmin=-1, vmax=1)
plt.title('Mapa de correlación entre variables', fontsize=14)
plt.tight_layout()
plt.show()

# Correlaciones más altas entre features (excluyendo target)
print('\nPares de variables con mayor correlación (|r| > 0.8):')
corr_features = corr.drop(target_col, axis=0).drop(target_col, axis=1) if target_col in corr.columns else corr
for i in range(len(corr_features.columns)):
    for j in range(i+1, len(corr_features.columns)):
        r = corr_features.iloc[i, j]
        if abs(r) > 0.8:
            print(f'  {corr_features.columns[i]} ↔ {corr_features.columns[j]}: r = {r:.3f}')

---
## 5. Resumen de Hallazgos Iniciales

A partir del análisis exploratorio y estadístico realizado, se identifican los siguientes hallazgos:

1. **Desbalance de clases:** aproximadamente el 22% de los clientes incumplió el pago (clase positiva). Esto implica que un clasificador trivial que prediga siempre "no incumple" alcanzaría un 78% de accuracy, métrica que resultaría engañosa. Por lo tanto, será necesario evaluar los modelos con métricas más robustas como ROC-AUC, F1-Score y recall.

2. **Límite de crédito:** los clientes que incumplen presentan, en promedio, límites de crédito significativamente más bajos que los cumplidores. El test t de Welch confirmó que esta diferencia es estadísticamente significativa (p < 0.001). Esto sugiere que la entidad ya poseía algún criterio de riesgo al asignar los límites.

3. **Nivel educativo:** los clientes con formación de posgrado o universitaria exhiben tasas de incumplimiento menores que aquellos con educación secundaria o clasificados como "Otros". Este patrón es consistente con la literatura sobre riesgo crediticio.

4. **Estado civil:** los clientes solteros presentan una tasa de incumplimiento levemente superior a los casados, aunque la diferencia es modesta.

5. **Historial de pagos (pay_0 a pay_6):** estas variables muestran la mayor correlación con la variable objetivo. El estado de pago más reciente (`pay_0`) es, individualmente, el predictor más fuerte. Esto refleja un principio conocido en análisis crediticio: el comportamiento reciente es el mejor indicador del comportamiento futuro.

6. **Multicolinealidad en montos facturados:** las variables `bill_amt1` a `bill_amt6` presentan correlaciones entre sí superiores a 0.8, lo que indica una marcada multicolinealidad. Esto se abordará en la etapa de selección de características para evitar redundancia en el modelo.

---
## 6. Definición del Problema y Selección del Modelo

### Definición formal del problema
**Objetivo:** construir un modelo de clasificación binaria supervisada que, a partir de las características demográficas y el historial crediticio de un cliente, prediga si incumplirá el pago de su tarjeta de crédito el mes siguiente (variable `default`: 0 = cumple, 1 = incumple).

### Modelos seleccionados

**Modelo principal: Random Forest Classifier**

Se elige Random Forest como modelo principal por las siguientes razones:
- Es robusto ante valores atípicos y no requiere normalización estricta de las variables.
- Maneja de forma nativa la combinación de variables categóricas codificadas y numéricas.
- Proporciona una medida de importancia de características (*feature importance*), lo cual facilita la interpretación del modelo para áreas de negocio.
- Reduce el riesgo de sobreajuste respecto a un árbol de decisión individual, al promediar múltiples árboles entrenados sobre submuestras aleatorias.

**Modelo baseline: Regresión Logística**

Se entrena adicionalmente una regresión logística como referencia de comparación. Este modelo lineal permite evaluar si las relaciones en los datos son suficientemente capturadas por un enfoque más simple, y establece un piso de rendimiento contra el cual juzgar al Random Forest.

---
## 7. Selección de Características

In [ ]:
# 7.1 Correlación de cada variable con la variable objetivo
corr_target = df.select_dtypes(include=[np.number]).corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 8))
colores = ['#F44336' if x > 0 else '#2196F3' for x in corr_target]
corr_target.plot(kind='barh', color=colores)
plt.title('Correlación de cada variable con el incumplimiento (default)', fontsize=13)
plt.xlabel('Coeficiente de correlación de Pearson')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('Top 10 variables más correlacionadas con default (en valor absoluto):')
for var, r in corr_target.head(10).items():
    print(f'  {var:20s}  r = {r:+.4f}')

In [ ]:
# 7.2 Selección de características para el modelo

# Se excluyen bill_amt2 a bill_amt6 por alta multicolinealidad con bill_amt1
# (correlaciones > 0.8 entre ellas). Se retiene bill_amt1 como representativa
# del nivel de endeudamiento mensual.

excluir = ['bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6']

features = [col for col in df.columns
            if col != target_col
            and col in df.select_dtypes(include=[np.number]).columns
            and col not in excluir]

print(f'Variables seleccionadas para el modelo ({len(features)}):' )
for f in features:
    print(f'  • {f}')

print(f'\nVariables excluidas ({len(excluir)}):')
for e in excluir:
    if e in df.columns:
        print(f'  ✗ {e}')

print()
print('Justificación: las variables bill_amt2 a bill_amt6 presentan correlación > 0.8')
print('entre sí, lo que introduce multicolinealidad. Se retiene bill_amt1 y se conservan')
print('todas las variables de estado de pago (pay_0 a pay_6) por ser los predictores')
print('individuales más fuertes según el análisis de correlación.')

---
## 8. Implementación y Entrenamiento del Modelo

In [ ]:
# 8.1 Preparación de los datos
X = df[features]
y = df[target_col]

# División estratificada 80% entrenamiento / 20% prueba
# La estratificación garantiza que ambos conjuntos mantengan la misma
# proporción de clases que el dataset original.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado (necesario para regresión logística, no afecta a Random Forest)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Conjunto de entrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Conjunto de prueba:        {X_test.shape[0]:,} registros ({X_test.shape[0]/len(df)*100:.0f}%)')
print(f'Proporción de default en train: {y_train.mean()*100:.1f}%')
print(f'Proporción de default en test:  {y_test.mean()*100:.1f}%')

In [ ]:
# 8.2 Modelo baseline: Regresión Logística
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('═' * 55)
print('  REGRESIÓN LOGÍSTICA (Baseline)')
print('═' * 55)
print(classification_report(y_test, y_pred_lr,
                            target_names=['No incumple', 'Incumple']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# 8.3 Modelo principal: Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('═' * 55)
print('  RANDOM FOREST')
print('═' * 55)
print(classification_report(y_test, y_pred_rf,
                            target_names=['No incumple', 'Incumple']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# 8.4 Validación cruzada (5-fold) para evaluar estabilidad
print('=== VALIDACIÓN CRUZADA (5-fold, métrica: ROC-AUC) ===')

cv_lr = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='roc_auc')
cv_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')

print(f'Regresión Logística:  AUC = {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'Random Forest:        AUC = {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
print()
print('La baja desviación estándar indica que los modelos generalizan de forma estable')
print('a través de diferentes particiones del conjunto de entrenamiento.')

In [ ]:
# 8.5 Matrices de confusión comparadas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, titulo in zip(axes,
                               [y_pred_lr, y_pred_rf],
                               ['Regresión Logística', 'Random Forest']):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['No incumple', 'Incumple'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(titulo, fontsize=13)

plt.suptitle('Matrices de confusión — Comparación de modelos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 8.6 Curvas ROC comparadas
plt.figure(figsize=(8, 6))

for y_proba, nombre, color, ls in [
    (y_proba_lr, 'Regresión Logística', '#2196F3', '-'),
    (y_proba_rf, 'Random Forest', '#F44336', '-')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{nombre} (AUC = {auc:.4f})', color=color, linewidth=2, linestyle=ls)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Clasificador aleatorio (AUC = 0.5)')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')
plt.xlabel('Tasa de falsos positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de verdaderos positivos (TPR)', fontsize=12)
plt.title('Curvas ROC — Comparación de modelos', fontsize=14)
plt.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# 8.7 Importancia de características (Random Forest)
importancias = pd.Series(rf.feature_importances_, index=features)
importancias_sorted = importancias.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colores_imp = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(importancias_sorted)))
importancias_sorted.plot(kind='barh', color=colores_imp)
plt.title('Importancia de características — Random Forest', fontsize=14)
plt.xlabel('Importancia relativa (Gini)')
plt.tight_layout()
plt.show()

print('Top 5 variables más importantes para el modelo:')
for var, imp in importancias.sort_values(ascending=False).head(5).items():
    print(f'  {var:20s}  {imp:.4f}')

---
## 9. Comunicación de Resultados

In [ ]:
# Tabla comparativa de métricas
resultados = pd.DataFrame({
    'Modelo': ['Regresión Logística (Baseline)', 'Random Forest (Principal)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision (Incumple)': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall (Incumple)': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score (Incumple)': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf)
    ]
}).round(4)

print('═' * 80)
print('  COMPARATIVA FINAL DE MODELOS')
print('═' * 80)
resultados.style.highlight_max(subset=['Accuracy', 'Precision (Incumple)',
                                       'Recall (Incumple)', 'F1-Score (Incumple)',
                                       'ROC-AUC'],
                                color='lightgreen')

### Interpretación de los resultados

El **Random Forest superó a la Regresión Logística** en las métricas más relevantes para este problema, particularmente en ROC-AUC y F1-Score de la clase minoritaria (incumplimiento).

**Lectura de las métricas en contexto bancario:**

- **Recall (Incumple):** representa la proporción de clientes incumplidores que el modelo logra detectar. Un recall bajo significaría que el banco no identifica a muchos morosos potenciales, lo que se traduce en pérdidas por créditos irrecuperables.
- **Precision (Incumple):** indica qué proporción de los clientes predichos como incumplidores efectivamente lo son. Una precisión baja implica rechazar clientes que en realidad son buenos pagadores, perdiendo oportunidades de negocio.
- **ROC-AUC:** métrica que evalúa la capacidad discriminativa del modelo independientemente del umbral de decisión. Es particularmente útil en datasets con desbalance de clases como el presente, donde el accuracy puede resultar engañoso.

El uso de `class_weight='balanced'` en ambos modelos permitió compensar parcialmente el desbalance de clases, priorizando la detección de la clase minoritaria sin necesidad de técnicas de remuestreo.

**Variable más predictiva:** el estado de pago más reciente (`pay_0`) resultó ser la variable de mayor importancia en el Random Forest, confirmando que el comportamiento de pago reciente es el mejor indicador del riesgo de incumplimiento futuro.

---
## 10. Propuestas de Implementación

### Aplicación práctica del modelo en un entorno productivo

1. **Sistema de scoring crediticio en tiempo real:** integrar el modelo entrenado en el flujo de aprobación de créditos, asignando un score de riesgo (probabilidad de incumplimiento) a cada solicitante. Los clientes cuya probabilidad supere un umbral definido por el área de riesgo (por ejemplo, 0.5) serían derivados a revisión manual antes de aprobar la operación.

2. **Monitoreo de cartera con alertas tempranas:** ejecutar el modelo mensualmente sobre la cartera activa de clientes para identificar aquellos que muestran señales de deterioro crediticio (aumento en los meses de retraso, reducción en los montos pagados). Esto permite la intervención proactiva —contacto telefónico, renegociación de deuda, ajuste de límites— antes de que se materialice el incumplimiento.

3. **Segmentación de riesgo para estrategias comerciales:** clasificar a los clientes en segmentos de riesgo (bajo, medio, alto) para personalizar la oferta de productos, los límites de crédito, las tasas de interés y la frecuencia de contacto. Los clientes de bajo riesgo podrían recibir aumentos de límite o promociones, mientras que los de alto riesgo recibirían un seguimiento más cercano.

4. **Mejoras futuras del modelo:**
   - **Balanceo de clases:** aplicar técnicas de sobremuestreo sintético (SMOTE) para mejorar el recall sin sacrificar excesivamente la precisión.
   - **Modelos de gradiente:** evaluar XGBoost o LightGBM, que suelen ofrecer mejor rendimiento en este tipo de problemas tabulares.
   - **Variables externas:** incorporar datos de buró de crédito, historial en otras entidades o variables macroeconómicas.
   - **Monitoreo de *drift*:** implementar métricas de seguimiento para detectar la degradación del modelo en el tiempo (concept drift).

5. **Consideraciones regulatorias:** en Argentina, los modelos de scoring deben cumplir con las normativas del BCRA sobre transparencia, no discriminación y protección de datos personales (Ley 25.326). Se recomienda incorporar técnicas de explicabilidad (SHAP values) para justificar las decisiones del modelo a nivel individual, facilitando tanto la auditoría interna como la respuesta a reclamos de clientes.

---

### Bibliografía
- Yeh, I. C., & Lien, C. H. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients*. Expert Systems with Applications, 36(2), 2473-2480.
- Material del curso — UTN BA Centro de E-Learning.
- Documentación oficial: scikit-learn (https://scikit-learn.org), pandas, matplotlib, seaborn.